# 01 — CDC WONDER suppression check (Ohio pilot)

**Question:** Is county-level CVD mortality from CDC WONDER usable, or is too much suppressed?

**Data:** CDC WONDER Underlying Cause of Death, 2018–2024 Single Race. Ohio, 2018–2022, ICD-10 I00–I99.
- A: by county (5 years pooled)
- B: by county × year
- C: by county × sex

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")

def load_wonder(path):
    """Load a CDC WONDER export, keeping only real data rows."""
    # dtype=str keeps FIPS codes as text so leading zeros survive
    df = pd.read_csv(path, sep="\t", dtype=str)
    # Drop the "Total" subtotal rows and the query-notes footer
    df = df[df["Notes"].isna() & df["County Code"].notna()].drop(columns="Notes")
    for col in ["Deaths", "Population", "Crude Rate"]:
        df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")
    return df

a = load_wonder(RAW / "A_ohio_county_2018_2022.txt")
b = load_wonder(RAW / "B_ohio_county_year.txt")
c = load_wonder(RAW / "C_ohio_county_sex.txt")

In [2]:
# Ohio has 88 counties; B has 5 years per county, C has 2 sexes per county
assert len(a) == 88, len(a)
assert len(b) == 88 * 5, len(b)
assert len(c) == 88 * 2, len(c)
a.head()

,County,County Code,Deaths,Population,Crude Rate,Crude Rate Lower 95% Confidence Interval,Crude Rate Upper 95% Confidence Interval,Deaths_num,Population_num,Crude Rate_num
0,"Adams County, OH",39001,545,137915,395.2,362.0,428.3,545,137915,395.2
1,"Allen County, OH",39003,2022,509779,396.6,379.4,413.9,2022,509779,396.6
2,"Ashland County, OH",39005,1045,265088,394.2,370.3,418.1,1045,265088,394.2
3,"Ashtabula County, OH",39007,2096,485598,431.6,413.2,450.1,2096,485598,431.6
4,"Athens County, OH",39009,792,317661,249.3,232.0,266.7,792,317661,249.3


In [11]:
def suppression_report(df, label):
    """Return % rows suppressed, % population in suppressed rows, % rows unreliable."""
    suppressed = df["Deaths"] == "Suppressed"
    unreliable = df["Crude Rate"] == "Unreliable"
    pct_rows_suppressed = suppressed.mean()
    pct_pop_suppressed = df.loc[suppressed, "Population_num"].sum() / df["Population_num"].sum()
    pct_rows_unreliable = unreliable.mean()
    return {"query": label,
            "pct_rows_suppressed": pct_rows_suppressed,
            "pct_pop_suppressed": pct_pop_suppressed,
            "pct_rows_unreliable": pct_rows_unreliable}

pd.DataFrame([suppression_report(a, "A: county"),
              suppression_report(b, "B: county × year"),
              suppression_report(c, "C: county × sex")])

,query,pct_rows_suppressed,pct_pop_suppressed,pct_rows_unreliable
0,A: county,0.0,0.0,0.0
1,B: county × year,0.0,0.0,0.0
2,C: county × sex,0.0,0.0,0.0


In [ ]:
#Confirming real zeros.

for name, df in [("A", a), ("B", b), ("C", c)]:
    smallest = df.loc[df["Deaths_num"].idxmin()]
    print(name, "min deaths:", smallest["Deaths_num"], "-", smallest["County"],
          "| non-numeric Deaths:", df["Deaths_num"].isna().sum())

A min deaths: 216 - Noble County, OH | non-numeric Deaths: 0
B min deaths: 36 - Vinton County, OH | non-numeric Deaths: 0
C min deaths: 98 - Noble County, OH | non-numeric Deaths: 0


In [ ]:
# Comparing yearly crude rates for two counties.

b[b["County"].isin(["Vinton County, OH", "Cuyahoga County, OH"])][
    ["County", "Year", "Deaths", "Crude Rate",
     "Crude Rate Lower 95% Confidence Interval",
     "Crude Rate Upper 95% Confidence Interval"]
]

,County,Year,Deaths,Crude Rate,Crude Rate Lower 95% Confidence Interval,Crude Rate Upper 95% Confidence Interval
102,"Cuyahoga County, OH",2018,4795,385.5,374.6,396.4
103,"Cuyahoga County, OH",2019,4623,374.3,363.5,385.1
104,"Cuyahoga County, OH",2020,4851,395.1,384.0,406.2
105,"Cuyahoga County, OH",2021,4669,373.7,363.0,384.4
106,"Cuyahoga County, OH",2022,4569,369.6,358.9,380.4
486,"Vinton County, OH",2018,36,274.0,191.9,379.3
487,"Vinton County, OH",2019,43,328.6,237.8,442.6
488,"Vinton County, OH",2020,51,393.2,292.7,516.9
489,"Vinton County, OH",2021,50,393.8,292.3,519.2
490,"Vinton County, OH",2022,48,382.0,281.7,506.5


## Interpretation

The pilot test suggests that **county-level CVD mortality in Ohio is unaffected by suppression** at these levels of detail (pooled 2018–2022, by year, and by sex; ICD-10 I00–I99).

However, this does not mean every county's rate is equally reliable. Smaller counties have far fewer deaths, not lower rates:

- **Vinton County:** as few as 36 deaths in a single year
- **Cuyahoga County (Cleveland):** 4795 deaths in the same year

Because their rates rest on so few deaths, smaller counties have much wider confidence intervals around their rates.

**Implication for design:** model death counts (Poisson/negative binomial with a population offset) rather than raw rates, so that larger, more precise counties carry more weight.